# BTXRD Rich Gallery G1 — end-to-end inference demo

Notebook này chạy **một ảnh** qua đúng pipeline suy luận đã chốt. Mỗi cell tương ứng một giai đoạn để có thể trình bày trực tiếp. Notebook chỉ dùng ảnh và nhãn ảnh tumor/normal; không đọc polygon hay mặt nạ chuẩn và không tính Dice/IoU.

## 0. Cấu hình

Mặc định chọn `IMG001661.jpeg`, một ảnh validation phù hợp để minh họa. Chỉ cần sửa ba đường dẫn bên dưới nếu máy hoặc dataset được đặt ở vị trí khác.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / 'project').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent.resolve()
sys.path.insert(0, str(REPOSITORY_ROOT))

from project.demo_final_pipeline import DemoConfig, show_demo

CHECKPOINT_ROOT = REPOSITORY_ROOT / 'checkpoints' / 'final_method'
DATASET_ROOT = Path(r'C:\path\to\BTXRD')       # sửa đường dẫn này
SPLIT_MANIFEST = Path(r'C:\path\to\canonical_split_manifest.csv')  # sửa đường dẫn này
IMAGE_ID = 'IMG001661.jpeg'
SPLIT = 'val'
WORK_DIR = REPOSITORY_ROOT / 'demo_outputs' / Path(IMAGE_ID).stem
FROZEN_TEST_CONFIG = None  # bắt buộc cung cấp nếu đổi SPLIT thành 'test'
def environment_python(name):
    windows = REPOSITORY_ROOT / name / 'Scripts' / 'python.exe'
    linux = REPOSITORY_ROOT / name / 'bin' / 'python'
    return windows if windows.exists() else linux

CANDIDATE_PYTHON = environment_python('.venv-candidate')
G1_PYTHON = environment_python('.venv-g1')
if not CANDIDATE_PYTHON.is_file() or not G1_PYTHON.is_file():
    raise FileNotFoundError('Create .venv-candidate and .venv-g1 exactly as described in docs/USAGE.md')

cfg = DemoConfig(
    repository_root=REPOSITORY_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    dataset_root=DATASET_ROOT,
    split_manifest=SPLIT_MANIFEST,
    split=SPLIT,
    image_id=IMAGE_ID,
    work_dir=WORK_DIR,
    frozen_config=FROZEN_TEST_CONFIG,
)
CONFIG_PATH = cfg.write_json(WORK_DIR / 'demo_config.json')

def run_stage(stage, python_executable=CANDIDATE_PYTHON):
    command = [str(python_executable), '-m', 'project.demo_final_pipeline', stage, '--config', str(CONFIG_PATH)]
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPOSITORY_ROOT), str(REPOSITORY_ROOT / 'project'), env.get('PYTHONPATH', '')])
    subprocess.run(command, cwd=REPOSITORY_ROOT, env=env, check=True)

cfg

## 1. Kiểm tra dữ liệu và checkpoint

Cell này xác minh ảnh theo split manifest và đối chiếu SHA-256 của toàn bộ checkpoint trước khi GPU được sử dụng.

In [ ]:
run_stage('verify')

## 2. BiomedCLIP: nhãn ảnh → bản đồ saliency

BiomedCLIP tạo bằng chứng định vị từ độ tương phản giữa prompt tumor và normal trên toàn ảnh cùng các crop cục bộ Top-3.

In [ ]:
run_stage('biomedclip')
biomedclip_result = json.loads((WORK_DIR / '01_biomedclip' / 'run_metadata.json').read_text())
biomedclip_result

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

image = Image.open(DATASET_ROOT / 'images' / IMAGE_ID).convert('RGB')
saliency = np.load(WORK_DIR / '01_biomedclip' / 'maps' / f'{Path(IMAGE_ID).stem}.npy', allow_pickle=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(image); axes[0].set_title('Input X-ray'); axes[0].axis('off')
axes[1].imshow(saliency, cmap='magma'); axes[1].set_title('BiomedCLIP saliency'); axes[1].axis('off')
plt.tight_layout()

## 3. Nguồn anchor: LayerCAM-320 + BiomedCLIP → SAM ViT-B

Classifier 320 sinh LayerCAM; LayerCAM và BiomedCLIP tạo point/box prompts. SAM ViT-B biến các prompt thành tập ứng viên mask.

In [ ]:
run_stage('anchor')
anchor_result = json.loads((WORK_DIR / '02_anchor' / 'candidate_diagnostics_summary.json').read_text())
anchor_result

## 4. Nguồn bổ sung: LayerCAM-448 → SAM ViT-B

Nhánh 448 giữ thêm chi tiết không gian và tạo một tập proposal độc lập bằng cùng SAM ViT-B.

In [ ]:
run_stage('addition')
addition_result = json.loads((WORK_DIR / '03_addition' / 'candidate_diagnostics_summary.json').read_text())
addition_result

## 5. Rich proposal gallery

Hai nguồn proposal được đưa về lưới anchor, gắn định danh nguồn và loại trùng theo mask nhị phân chính xác.

In [ ]:
run_stage('merge')
gallery_result = json.loads((WORK_DIR / '04_merged_gallery' / 'candidate_diagnostics_summary.json').read_text())
gallery_result

## 6. RAD-DINO + G1 + equal percentile-rank fusion

RAD-DINO trích đặc trưng bên trong, lân cận và độ tương phản cho từng mask. G1 cho logit từng ứng viên. Mask cuối là argmax ổn định của trung bình percentile-rank giữa G1 và upstream score.

In [ ]:
run_stage('score', G1_PYTHON)
final_result = json.loads((WORK_DIR / '05_final' / 'demo_receipt.json').read_text())
final_result

## 7. Kết quả cuối

Hình chỉ hiển thị artifact suy luận. Ground-truth không được mở trong notebook demo.

In [ ]:
figure = show_demo(cfg)
figure

In [ ]:
import pandas as pd
scores = pd.read_csv(final_result['candidate_scores'])
scores.sort_values(['selected', 'fused_rank_score'], ascending=False).head(10)